# Module 3: Security Scanning & The Safety Net

**Course:** UNDP IICPSD Agentic AI & Vibe Coding Training Series 2026 | **Instructor:** Phyo Ko Ko, Senior Cyber Security Analyst

---

## Why This Module Matters

AI agents write code fast - sometimes dangerously fast. Without a validation layer, vibe coding commonly produces:

- **Hardcoded secrets** - API keys, passwords, and tokens written directly into source files
- **SQL injection vulnerabilities** - user input inserted raw into database queries
- **Dangerous functions** - use of `eval()` and `exec()` that execute arbitrary code
- **Debug flags left on** - `DEBUG = True` shipped to production environments
- **Insecure dependencies** - malicious or outdated third-party packages

The **Safety Net** is a 3-pillar architecture that catches these issues before they reach production.

## The 3-Pillar Safety Net Architecture

```
[Pillar 1: Static Guardrails]
 - .cursorrules blocks bad patterns at code generation time
 - Pre-commit hooks (TruffleHog, Bandit) scan before git commit
 - Catches: hardcoded secrets, dangerous functions, insecure imports
        |
        v
[Pillar 2: Dynamic Evals]
 - AI writes unit tests for its own generated code
 - Tests run automatically in CI/CD pipeline
 - Catches: logic errors, edge cases, runtime failures
        |
        v
[Pillar 3: Model-Graded Evals]
 - A second high-reasoning LLM audits the generated code
 - Acts as an independent compliance reviewer
 - Catches: architectural flaws, security anti-patterns
```


## Key Terms

| Term | Definition |
|---|---|
| **Static Analysis** | Examining source code without executing it to find vulnerabilities (e.g. Bandit, TruffleHog). |
| **Dynamic Analysis** | Testing code by executing it and observing runtime behavior (e.g. unit tests, fuzz testing). |
| **Hardcoded Secret** | A credential written directly into source code instead of loaded from environment variables. |
| **SQL Injection** | An attack where user input is inserted raw into a SQL query, allowing database manipulation. |
| **Pre-commit Hook** | A script that runs automatically before a git commit, blocking it if security issues are found. |
| **DevSecOps** | The practice of integrating security checks directly into the development and deployment pipeline. |


---

## Lab 3: The Safety Net (Automated Security Scanner)

### Objective

Build a static analysis scanner that simulates TruffleHog and Bandit. It inspects AI-generated Python code at runtime and flags hardcoded credentials before they can be committed to source control.


In [ ]:
import re
import inspect

# Simulated output from an AI agent that generated code without security rules
def ai_generated_vibe_code():
    session_token = "AWS_SECRET_ACCESS_KEY_XYZ1234567890ABCDEFG"
    cloud_provider = "AWS"
    print(f"Connecting to {cloud_provider} cloud service...")
    return True

def run_safety_net_scanner(target_code_function):
    print("Deploying Pillar 1: Automated Static Guardrail Scan...")

    # Extract the raw source code of the function as a string
    source_lines = inspect.getsource(target_code_function)

    # Regex pattern to detect hardcoded credentials
    secret_pattern = re.compile(r'(?i)(key|secret|token|password)\s*=\s*[\'"][^\'"]{15,}[\'"]')
    findings = secret_pattern.findall(source_lines)

    print("\n--- Safety Net Audit Report ---")
    if findings:
        print(f"  CRITICAL VULNERABILITY DETECTED!")
        print(f"  Hardcoded credentials found: {findings}")
        print("  Status: Commit Blocked. Refactor using environment variables.")
        return False
    else:
        print("  Status: Pass. No hardcoded secrets detected.")
        return True

run_safety_net_scanner(ai_generated_vibe_code)


## What Just Happened?

`inspect.getsource()` extracts the raw Python source of the function as a plain string. The regex pattern then searches for variable assignments that look like credentials - any variable named `key`, `secret`, `token`, or `password` assigned a string longer than 15 characters.

This is exactly how **TruffleHog** works in production - scanning source text for patterns matching known secret formats across 800+ credential types. In a real pipeline, this scanner runs as a **pre-commit hook**: before `git commit` is allowed to complete, the hook runs and blocks the commit entirely if secrets are found.


---

## Assignment 3

### Part A: Reflection Questions

**Q1.** Name three types of security vulnerabilities that AI-generated code commonly introduces beyond hardcoded credentials. Briefly explain each.

> *Your answer here...*

**Q2.** What is the difference between static analysis (like Bandit) and dynamic analysis for security scanning? Give one example use case for each.

> *Your answer here...*

**Q3.** The Safety Net has 3 pillars. In your own words, describe what each pillar does and name one real tool that represents it.

> *Your answer here...*

### Part B: Coding Exercise - Extend the Security Scanner

The Lab 3 scanner only detects hardcoded credentials. Extend it in the cell below to also detect:

1. **Dangerous functions** - any use of `eval()` or `exec()` in the code
2. **SQL injection risk** - f-strings used inside queries containing `SELECT`, `INSERT`, `UPDATE`, or `DELETE`
3. **Debug flags** - `DEBUG = True` left in the code

Each finding should appear as a separate line in the audit report.


In [ ]:
import re
import inspect

# Sample AI-generated code containing all 3 vulnerability types
def vulnerable_ai_code():
    db_password = "SuperSecret123!@#"
    DEBUG = True
    user_input = input("Enter user ID: ")
    query = f"SELECT * FROM users WHERE id = {user_input}"
    result = eval(query)
    return result

def extended_safety_scanner(target_function):
    print("Deploying Extended Safety Net Scanner...")
    source = inspect.getsource(target_function)
    findings = []

    # Existing check: hardcoded credentials
    cred_pattern = re.compile(r'(?i)(key|secret|token|password)\s*=\s*[\'"][^\'"]{8,}[\'"]')
    if cred_pattern.findall(source):
        findings.append("CRITICAL - Hardcoded credentials detected")

    # TODO 1: Detect eval() or exec()
    # Hint: search for the word eval or exec followed by (

    # TODO 2: Detect SQL injection risk (f-string with SELECT/INSERT/UPDATE/DELETE)
    # Hint: look for f" or f' followed by a SQL keyword

    # TODO 3: Detect DEBUG = True
    # Hint: case-insensitive match for debug followed by = True

    print("\n--- Safety Net Audit Report ---")
    if findings:
        for f in findings:
            print(f"  [!] {f}")
        print("\n  Status: BLOCKED - Fix all issues before deployment.")
        return False
    else:
        print("  Status: PASS - No vulnerabilities detected.")
        return True

extended_safety_scanner(vulnerable_ai_code)
